In [1]:
import os
import rasterio
import logging
import rioxarray as rio
import numpy as np

In [19]:
#folders and filepaths
imagery_folder = '../../../data/raw/satellite'

In [3]:
for file in os.listdir(imagery_folder):
    if file.lower().endswith('.tif'):
        input_path = os.path.join(imagery_folder, file)
        with rasterio.open(input_path) as src:
            print(src.descriptions)


('SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7')
('SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7')
('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12')
('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12')


In [ ]:
def explode_geotiff(input_dir):
    '''
    Finds all multiband geotiffs in a folder and explodes them into individual bands
    :param input_dir: The directory where the geotiffs are housed
    '''
    for file in os.listdir(input_dir):
        if not file.lower().endswith('.tif') or file.lower().contains('B'):
            continue
        input_path = os.path.join(input_dir, file)
        base_name = os.path.splitext(file)[0]

        with rasterio.open(input_path) as src:
            profile = src.profile

            for band_index in range(1, src.count + 1):
                band_data = src.read(band_index)

                # 🔑 Get existing band name (authoritative)
                band_name = src.descriptions[band_index - 1]

                if band_name is None:
                    raise ValueError(
                        f"Band {band_index} in {file} has no name — "
                        "metadata is inconsistent with expectations."
                    )

                safe_band_name = band_name.replace(" ", "_")

                output_name = f"{base_name}_{safe_band_name}.tif"
                output_path = os.path.join(input_dir, output_name)

                profile.update(count=1)

                with rasterio.open(output_path, "w", **profile) as dst:
                    dst.write(band_data, 1)

                    # 🔑 Preserve band description
                    dst.set_band_description(1, band_name)

        print(f"Exploded with preserved band names: {file}")

In [5]:
explode_geotiff(imagery_folder)

Exploded with preserved band names: landsat5_2010.tif
Exploded with preserved band names: landsat8_2015.tif
Exploded with preserved band names: sentinel2_2020.tif
Exploded with preserved band names: sentinel2_2025.tif


In [12]:
def find_band(input_dir, target_band: str, year: int):
        '''
        Finds a specific band in a directory containing satellite imagery bands
        :param target_band: this is a string containing the band desc to target
        :returns: path of the band matching the regex
        >>> find_band('b2')
        data/processed/sentinel2_b2.tif
        '''
        for file in os.listdir(input_dir):
            if str(year) not in file.lower():
                continue
            if target_band in file and file.lower().endswith('.tif'):
                band_path = os.path.join(input_dir, file)
                print(f'Found band {band_path}')
                return band_path
        raise FileNotFoundError(f'Could not find any specific band matching the {target_band}.')

In [13]:
find_band(imagery_folder, 'SR_B2', 2010)

Found band ../../../data/raw/satellite\landsat5_2010_SR_B2.tif


'../../../data/raw/satellite\\landsat5_2010_SR_B2.tif'

In [ ]:
def normalized_difference(input_dir, first: str, next: str, year, index: str):
        '''
        This is a function to calculate the normalized difference between bands
        :param first: The first band to be used in equation- key in a desc('b2')
        :param next: The second band to be used in equation- key in a desc('b8')
        :returns: Normalized difference tif
        '''
        first_path = find_band(input_dir, first, year)
        next_path = find_band(input_dir, next, year)
        out_path = os.path.join(input_dir, f'{year}_{index}.tif')
        
        with rasterio.open(first_path) as band, rasterio.open(next_path) as band0:
            if band.shape != band0.shape or band.crs !=band0.crs:
                raise ValueError('Ensure crs and shape align for rasters')
            red = band.read(1).astype('float32')
            nir = band0.read(1).astype('float32')
            # Avoid division by zero
            np.seterr(divide='ignore', invalid='ignore')
            ndvi = (red - nir) / (red + nir)
            ndvi = np.clip(ndvi, -1, 1)
            meta = band.meta.copy()
            meta.update({
            "count": 1,
            "dtype": "float32",
            "driver": "GTiff",
            "nodata": -9999
            })
            print(f'Writing the index to disk: {out_path}')
            with rasterio.open(out_path, 'w', **meta) as dest:
                dest.write(ndvi, 1)
        return out_path

In [17]:
def bsi(input_dir, year):
        '''
        This is function to calculate the bare soil index
        '''
        out_path = os.path.join(input_dir, f'{year}_bsi.tif')
        try:
            if str(year) == '2020' or str(year) == '2025':
                swir = find_band(input_dir, 'B11', year)
                red = find_band(input_dir, 'B4', year)
                nir = find_band(input_dir, 'B8', year)
                blue = find_band(input_dir, 'B2', year)
            elif str(year) == '2015':
                swir = find_band(input_dir, 'SR_B6', year)
                red = find_band(input_dir, 'SR_B4', year) 
                nir = find_band(input_dir, 'SR_B5', year) 
                blue = find_band(input_dir, 'SR_B2', year) 
            elif str(year) == '2010':
                swir = find_band(input_dir, 'SR_B5', year)
                red = find_band(input_dir, 'SR_B3', year) 
                nir = find_band(input_dir, 'SR_B4', year) 
                blue = find_band(input_dir, 'SR_B1', year) 
            else:
                'Check your year input'
        except:
            raise KeyError('Error')
        
        with rasterio.open(swir) as swir_src, rasterio.open(red) as red_src, rasterio.open(nir) as nir_src, rasterio.open(blue) as blue_src:
            if swir_src.shape != red_src.shape != nir_src.shape != blue_src.shape  or swir_src.crs != red_src.crs != nir_src.crs != blue_src.crs:
                raise ValueError('Ensure crs and shape align for rasters')
            swir = swir_src.read(1).astype('float32')
            red = red_src.read(1).astype('float32')
            nir = nir_src.read(1).astype('float32')
            blue = blue_src.read(1).astype('float32')
            np.seterr(divide='ignore', invalid='ignore')
            bsi = ((swir + red) - (nir +  blue))/((swir + red) + (nir + blue))
            bsi = np.clip(bsi, -1, 1)
            meta = swir_src.meta.copy()
            meta.update({
                "count": 1,
                "dtype": "float32",
                "driver": "GTiff",
                "nodata": -9999
            })
            print(f'Writing bsi to disk: {out_path}')
            with rasterio.open(out_path, 'w', **meta) as dest:
                dest.write(bsi, 1)

In [15]:
normalized_difference(imagery_folder, 'SR_B2', 'SR_B5', 2010, 'NDMI')

Found band ../../../data/raw/satellite\landsat5_2010_SR_B2.tif
Found band ../../../data/raw/satellite\landsat5_2010_SR_B5.tif
Writing the index to disk: ../../../data/raw/satellite\2010_index.tif


'../../../data/raw/satellite\\2010_index.tif'

In [18]:
bsi(imagery_folder, 2010)

Found band ../../../data/raw/satellite\landsat5_2010_SR_B5.tif
Found band ../../../data/raw/satellite\landsat5_2010_SR_B3.tif
Found band ../../../data/raw/satellite\landsat5_2010_SR_B4.tif
Found band ../../../data/raw/satellite\landsat5_2010_SR_B1.tif
Writing bsi to disk: ../../../data/raw/satellite\2010_bsi.tif
